要运行此程序，请在**免费** Tesla T4 Google Colab 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  #  Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install -qU llama-index llama-index-packs-raft-dataset

### 不懒惰

### 检索增强微调 (RAFT) 食谱！
本食谱旨在展示如何使用 Unsloth 来使用检索增强微调 (RAFT)。监督微调就像闭卷考试，我们在微调期间将训练数据集中的知识编码到 LLM 中，然后在“考试”中用未见过的示例进行测试。

RAFT与此不同的是，它是微调的开卷考试形式！我们让法学硕士不仅可以看到问题和答案（以思想链的形式），还可以看到上下文。希望法学硕士能够获得领域知识，同时提高从上下文中综合答案的能力。

> 参考：[RAFT: Adapting Language Model to Domain Specific RAG](https://arxiv.org/abs/2403.10131)

### 代码设置

首先，让我们设置 OPENAI API KEY，以便我们可以使用 OpenAI LLM。

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

接下来，我们将设置 LlamaIndex。这涉及配置 LlamaIndex 将使用的语言模型 (LLM) 和嵌入模型。我们将使用 OpenAI 的“gpt-4o”作为我们的 LLM，使用“text-embedding-ada-002”作为我们的嵌入模型。

In [ ]:
from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model = "gpt-4o")
Settings.embed_model = OpenAIEmbedding(model = "text-embedding-ada-002")

### 摄取文档

我们将使用以下代码下载研究论文，然后使用“SimpleDirectoryReader”加载它。这将是我们用于检索增强微调的数据。

In [ ]:
!mkdir  -p ../data
!wget "https://arxiv.org/pdf/2405.00247.pdf" -O "../data/non_traditional_credentials.pdf"

docs = SimpleDirectoryReader("../data/").load_data(show_progress = True)

Loading files:   0%|          | 0/1 [00:00<?, ?file/s]

Loading files: 100%|██████████| 1/1 [00:00<00:00,  1.19file/s]

### 检索增强微调

### 获取RAFT数据集
LlamaIndex 非常友善地改编了 RAFT 存储库的源代码，使生成您自己的 RAFT 数据集变得更加容易。只需将其指向您的 filepath.t
> 参考：[RAFTDatasetPack](https://github.com/run-llama/llama_index/blob/main/llama-index-packs/llama-index-packs-raft-dataset/examples/raft_dataset.ipynb)

In [14]:
from llama_index.packs.raft_dataset import RAFTDatasetPack

raft_dataset = RAFTDatasetPack(
    file_path = "../data/non_traditional_credentials.pdf",
    llm = Settings.llm,
    embed_model = Settings.embed_model
)

该单元需要很长时间才能运行！去喝杯咖啡吧☕
> 单元运行完成需要 19 分钟

In [15]:
dataset = raft_dataset.run()

我们来看看吧！

In [18]:
import pandas as pd
df = pd.DataFrame(dataset)
df.head()

,id,type,question,context,oracle_context,cot_answer,instruction
0,seed_task_0,general,What percentage increase in credential sharing...,{'sentences': [['The value of non-traditional ...,The value of non-traditional credentials in th...,assistant: To determine the percentage increas...,<DOCUMENT>The value of non-traditional credent...
1,seed_task_1,general,How much more likely were learners in the trea...,{'sentences': [['The control group did not rec...,The value of non-traditional credentials in th...,"assistant: To answer the question ""How much mo...",<DOCUMENT>The control group did not receive th...
2,seed_task_2,general,What was the increase in jobs related to the c...,{'sentences': [['The value of non-traditional ...,The value of non-traditional credentials in th...,assistant: To determine the increase in jobs r...,<DOCUMENT>The value of non-traditional credent...
3,seed_task_3,general,Which group of LinkedIn users showed a more pr...,{'sentences': [['The value of non-traditional ...,The value of non-traditional credentials in th...,assistant: To determine which group of LinkedI...,<DOCUMENT>The value of non-traditional credent...
4,seed_task_4,general,What platform were the courses completed on fo...,"{'sentences': [['Analogously, Past Managerial ...",The value of non-traditional credentials in th...,"assistant: To answer the question ""What platfo...","<DOCUMENT>Analogously, Past\nManagerial Job fo..."


In [24]:
from IPython.display import display, Markdown

display(Markdown(df.iloc[0]['instruction']))

<DOCUMENT>The value of non-traditional credentials in the labor market*
Susan Athey & Emil Palikot
May 2, 2024
Abstract
This study investigates the labor market value of credentials obtained from Massive Open On-
line Courses (MOOCs) and shared on business networking platforms. We conducted a random-
ized experiment involving more than 800,000 learners, primarily from developing countries and
without college degrees, who completed technology or business-related courses on the Coursera
platform between September 2022 and March 2023. The intervention targeted learners who had
recently completed their courses, encouraging them to share their credentials and simplifying the
sharing process. One year after the intervention, we collected data from LinkedIn profiles of ap-
proximately 40,000 experimental subjects. We find that the intervention leads to an increase of 17
percentage points for credential sharing. Further, learners in the treatment group were 6% more
likely to report new employment within a year, with an 8% increase in jobs related to their certifi-
cates. This effect was more pronounced among LinkedIn users with lower baseline employability.
Across the entire sample, the treated group received a higher number of certificate views, indicat-
ing an increased interest in their profiles. These results suggest that facilitating credential sharing
and reminding learners of the value of skill signaling can yield significant gains. When the ex-
periment is viewed as an encouragement design for credential sharing, we can estimate the local
average treatment effect (LATE) of credential sharing (that is, the impact of credential sharing on
the workers induced to share by the intervention) for the outcome of getting a job. The LATE esti-
mates are imprecise but large in magnitude; they suggest that credential sharing more than doubles
the baseline probability of getting a new job in scope for the credential.
*We thank Eric Karsten and his team in Coursera for collaborating on this project. </DOCUMENT>
<DOCUMENT>13 p.p.) and 36 p.p. (S.E. </DOCUMENT>
<DOCUMENT>), which corresponds to a
17% increase from baseline. The remaining columns present estimates from the instrumental variable
regression with New Job and New Job in Scope as outcomes. In Columns 6, 7, and 8, we restrict attention
to jobs reported with a starting date at least four months after treatment. We estimate positive and
statistically significant effects. Specifically, we estimate the local average treatment effect of 0.24 (S.E.
0.13) for any new job starting at least one month after treatment and 0.36 (S.E. 0.12) when restricting
14</DOCUMENT>
What percentage increase in credential sharing was observed after the intervention?

In [27]:
display(Markdown(df.iloc[0]['oracle_context']))

The value of non-traditional credentials in the labor market*
Susan Athey & Emil Palikot
May 2, 2024
Abstract
This study investigates the labor market value of credentials obtained from Massive Open On-
line Courses (MOOCs) and shared on business networking platforms. We conducted a random-
ized experiment involving more than 800,000 learners, primarily from developing countries and
without college degrees, who completed technology or business-related courses on the Coursera
platform between September 2022 and March 2023. The intervention targeted learners who had
recently completed their courses, encouraging them to share their credentials and simplifying the
sharing process. One year after the intervention, we collected data from LinkedIn profiles of ap-
proximately 40,000 experimental subjects. We find that the intervention leads to an increase of 17
percentage points for credential sharing. Further, learners in the treatment group were 6% more
likely to report new employment within a year, with an 8% increase in jobs related to their certifi-
cates. This effect was more pronounced among LinkedIn users with lower baseline employability.
Across the entire sample, the treated group received a higher number of certificate views, indicat-
ing an increased interest in their profiles. These results suggest that facilitating credential sharing
and reminding learners of the value of skill signaling can yield significant gains. When the ex-
periment is viewed as an encouragement design for credential sharing, we can estimate the local
average treatment effect (LATE) of credential sharing (that is, the impact of credential sharing on
the workers induced to share by the intervention) for the outcome of getting a job. The LATE esti-
mates are imprecise but large in magnitude; they suggest that credential sharing more than doubles
the baseline probability of getting a new job in scope for the credential.
*We thank Eric Karsten and his team in Coursera for collaborating on this project. 

In [16]:
# 另存为 .jsonl 格式
dataset.to_json("raft_train.jsonl")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2966201

### 培训法学硕士
我们的数据集是一个 HuggingFace `Dataset` 对象，因此我们可以利用抽象的优势来进行训练-测试分割

In [19]:
splits = dataset.train_test_split(test_size = 0.1)
train_ds = splits["train"]
eval_ds  = splits["test"]

In [20]:
train_ds, eval_ds

(Dataset({
     features: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction'],
     num_rows: 301
 }),
 Dataset({
     features: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction'],
     num_rows: 34
 }))

### 现在让我们获取模型！

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048, # 选择任何一个用于长上下文！
    load_in_4bit = True,  # 4 位量化以减少内存
    load_in_8bit = False, 
    full_finetuning = False, 
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-21 06:09:36 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 05-21 06:09:36 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.4.7: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [22]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # 支持任意，但 = 0 已优化
    bias = "none",    # 支持任何，但=“无”已优化
    # [新]“unsloth”使用的 VRAM 减少了 30%，适合 2 倍大的批量大小！
    use_gradient_checkpointing = "unsloth", # 对于很长的上下文来说是真实的或“不懒惰的”
    random_state = 2025,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
)

Unsloth 2025.4.7 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


## 设置提示格式
我们需要将所有内容放在一个“文本”字段中，以便法学硕士接受培训。根据[RAFT paper](https://arxiv.org/abs/2403.10131)，我们添加了上下文以及问题和思路答案，以帮助我们的法学硕士学习如何使用上下文来回答问题。让我们这样做吧！

In [25]:
def formatting_prompts_func(examples):
    """Define a formatter that injects the retrieved context:"""
    
    texts = []
    for qn, ctx, oracle, instr, ans in zip(
        examples['question'],
        examples["context"],
        examples["oracle_context"],
        examples["instruction"],
        examples["cot_answer"]
    ):
        # 您可以选择使用“oracle_context”（黄金）与“context”（检索）
        # 这里我们展示了两者，但你可以只使用“context”。
        prompt = (
            "# ## 问题：\n"
            f"{qn}\n\n"
            "# ## 上下文：\n"
            f"{ctx}\n\n"
            "# ##（甲骨文段落）：\n"
            f"{oracle}\n\n"
            "# ## 说明：\n"
            f"{instr}\n\n"
            "# ## 答案：\n"
        )
        # 附加黄金答案加上 EOS
        texts.append(prompt + ans + tokenizer.eos_token)
    return {"text": texts}

# 然后：
train_ds = train_ds.map(formatting_prompts_func, batched = True)
eval_ds = eval_ds.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

让我们看看我们刚刚做了什么！

In [26]:
from IPython.display import display, Markdown

display(Markdown(pd.DataFrame(train_ds).head()['text'].iloc[0]))

### Question:
What is the mean value for the 'Data Science' variable in the LinkedIn matched sample?

### Context:
{'sentences': [['Table 1: Summary statistics pretreatment and outcome variables\nCoursera Internal Data LinkedIn Matched Sample\nVariable name Mean S.E. Mean S.E.\nTreatment 0.499 0.001 0.500 0.003\nPanel A: Pre-treatment covariates\nProfessional Experience Years – – 3.040 0.028\nPast Tech Job – – 0.127 0.002\nPast Managerial Job – – 0.064 0.001\nMain Skill Absolute 0.099 0.001 2.074 0.010\nMain Skill Standardized 0.000 <0.001 0.000 0.001\nComputer Science 0.252 0.001 0.230 0.002\nData Science 0.236 0.001 0.300 0.002\nInformation Technology 0.140 0.001 0.138 0.002\nGuided Project 0.168 0.001 0.097 0.002\nProfessional Certificate 0.005 <0.001 0.005 <0.001\nSpecialization 0.009 <0.001 0.009 0.001\nDeveloping Country 0.896 0.001 0.850 0.002\nAssociate Degree 0.029 <0.001 0.062 0.001\nBachelor Degree 0.127 0.001 0.367 0.003\nSome College 0.072 0.001 0.130 0.002\nDoctorate Degree 0.004 <0.001 0.012 0.001\nHigh School Diploma 0.059 0.001 0.097 0.002\nLess than High School 0.009 <0.001 0.012 0.001\nMasters Degree 0.050 0.001 0.146 0.002\nNo Education Mentioned 0.645 0.002 0.164 0.002\nProfessional Degree 0.004 <0.001 0.010 0.001\nMale 0.302 0.002 0.674 0.002\nGender Not Mentioned 0.533 0.002 0.101 0.002\nPanel B: Outcome variables\nNew Job – – 0.177 0.002\nNew Job in Scope – – 0.133 0.002\nCredential Shared – – 0.181 0.002\nAll Views 0.191 0.001 0.429 0.003\nAll Views by Others 0.143 0.001 0.318 0.002\nViews LinkedIn 0.165 0.001 0.409 0.003\nViews LinkedIn by Others 0.124 0.001 0.296 0.002\nNote: Professional Experience Years is the number of years between the starting date of the first job and August 2023. Past Tech Job\ntakes the value of 1 when the learner had a job title related to technology before randomization and zero otherwise. ', 'effects between the bottom and top tertiles, the difference is 0.1 p.p. (S.E. ', 'For each learner, Coursera assesses skill mastery and assigns a score (Red-\ndick, 2019). Additionally, we compute a max-mean standardization of the learners’ skill level. We also\nobserve the country where the learner registered for the course. Following the OECD classification,\nwe use this information to group countries into developing and developed. Finally, we also observe\nthe information provided by the learners in their registration survey. ']], 'title': [['placeholder_title', 'placeholder_title', 'placeholder_title']]}

### (Oracle Passages):
Table 1: Summary statistics pretreatment and outcome variables
Coursera Internal Data LinkedIn Matched Sample
Variable name Mean S.E. Mean S.E.
Treatment 0.499 0.001 0.500 0.003
Panel A: Pre-treatment covariates
Professional Experience Years – – 3.040 0.028
Past Tech Job – – 0.127 0.002
Past Managerial Job – – 0.064 0.001
Main Skill Absolute 0.099 0.001 2.074 0.010
Main Skill Standardized 0.000 <0.001 0.000 0.001
Computer Science 0.252 0.001 0.230 0.002
Data Science 0.236 0.001 0.300 0.002
Information Technology 0.140 0.001 0.138 0.002
Guided Project 0.168 0.001 0.097 0.002
Professional Certificate 0.005 <0.001 0.005 <0.001
Specialization 0.009 <0.001 0.009 0.001
Developing Country 0.896 0.001 0.850 0.002
Associate Degree 0.029 <0.001 0.062 0.001
Bachelor Degree 0.127 0.001 0.367 0.003
Some College 0.072 0.001 0.130 0.002
Doctorate Degree 0.004 <0.001 0.012 0.001
High School Diploma 0.059 0.001 0.097 0.002
Less than High School 0.009 <0.001 0.012 0.001
Masters Degree 0.050 0.001 0.146 0.002
No Education Mentioned 0.645 0.002 0.164 0.002
Professional Degree 0.004 <0.001 0.010 0.001
Male 0.302 0.002 0.674 0.002
Gender Not Mentioned 0.533 0.002 0.101 0.002
Panel B: Outcome variables
New Job – – 0.177 0.002
New Job in Scope – – 0.133 0.002
Credential Shared – – 0.181 0.002
All Views 0.191 0.001 0.429 0.003
All Views by Others 0.143 0.001 0.318 0.002
Views LinkedIn 0.165 0.001 0.409 0.003
Views LinkedIn by Others 0.124 0.001 0.296 0.002
Note: Professional Experience Years is the number of years between the starting date of the first job and August 2023. Past Tech Job
takes the value of 1 when the learner had a job title related to technology before randomization and zero otherwise. 

### Instruction:
<DOCUMENT>Table 1: Summary statistics pretreatment and outcome variables
Coursera Internal Data LinkedIn Matched Sample
Variable name Mean S.E. Mean S.E.
Treatment 0.499 0.001 0.500 0.003
Panel A: Pre-treatment covariates
Professional Experience Years – – 3.040 0.028
Past Tech Job – – 0.127 0.002
Past Managerial Job – – 0.064 0.001
Main Skill Absolute 0.099 0.001 2.074 0.010
Main Skill Standardized 0.000 <0.001 0.000 0.001
Computer Science 0.252 0.001 0.230 0.002
Data Science 0.236 0.001 0.300 0.002
Information Technology 0.140 0.001 0.138 0.002
Guided Project 0.168 0.001 0.097 0.002
Professional Certificate 0.005 <0.001 0.005 <0.001
Specialization 0.009 <0.001 0.009 0.001
Developing Country 0.896 0.001 0.850 0.002
Associate Degree 0.029 <0.001 0.062 0.001
Bachelor Degree 0.127 0.001 0.367 0.003
Some College 0.072 0.001 0.130 0.002
Doctorate Degree 0.004 <0.001 0.012 0.001
High School Diploma 0.059 0.001 0.097 0.002
Less than High School 0.009 <0.001 0.012 0.001
Masters Degree 0.050 0.001 0.146 0.002
No Education Mentioned 0.645 0.002 0.164 0.002
Professional Degree 0.004 <0.001 0.010 0.001
Male 0.302 0.002 0.674 0.002
Gender Not Mentioned 0.533 0.002 0.101 0.002
Panel B: Outcome variables
New Job – – 0.177 0.002
New Job in Scope – – 0.133 0.002
Credential Shared – – 0.181 0.002
All Views 0.191 0.001 0.429 0.003
All Views by Others 0.143 0.001 0.318 0.002
Views LinkedIn 0.165 0.001 0.409 0.003
Views LinkedIn by Others 0.124 0.001 0.296 0.002
Note: Professional Experience Years is the number of years between the starting date of the first job and August 2023. Past Tech Job
takes the value of 1 when the learner had a job title related to technology before randomization and zero otherwise. </DOCUMENT>
<DOCUMENT>effects between the bottom and top tertiles, the difference is 0.1 p.p. (S.E. </DOCUMENT>
<DOCUMENT>For each learner, Coursera assesses skill mastery and assigns a score (Red-
dick, 2019). Additionally, we compute a max-mean standardization of the learners’ skill level. We also
observe the country where the learner registered for the course. Following the OECD classification,
we use this information to group countries into developing and developed. Finally, we also observe
the information provided by the learners in their registration survey. </DOCUMENT>
What is the mean value for the 'Data Science' variable in the LinkedIn matched sample?

### Answer:
assistant: To answer the question about the mean value for the 'Data Science' variable in the LinkedIn matched sample, we need to follow these steps:

1. **Identify the Relevant Section**: The context provides a table with summary statistics for various variables. We need to locate the section that pertains to the LinkedIn matched sample.

2. **Locate the 'Data Science' Variable**: Within the table, find the row labeled 'Data Science'. This row will contain the mean value for the 'Data Science' variable in both the Coursera Internal Data and the LinkedIn Matched Sample.

3. **Extract the Mean Value for LinkedIn Matched Sample**: Once the 'Data Science' row is located, extract the mean value specifically for the LinkedIn matched sample.

Let's proceed with these steps:

- The context provides a table with two columns of mean values: one for Coursera Internal Data and another for LinkedIn Matched Sample.

- ##begin_quote## Data Science 0.236 0.001 0.300 0.002 ##end_quote##: This line from the context shows the mean values for the 'Data Science' variable. The first mean value (0.236) corresponds to the Coursera Internal Data, and the second mean value (0.300) corresponds to the LinkedIn Matched Sample.

Therefore, the mean value for the 'Data Science' variable in the LinkedIn matched sample is 0.300.

<ANSWER>: 0.300<|eot_id|>

### 现在我们终于开始训练了！

In [28]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "llama32_1bn_raft_v2", # 这也将用作您的 Huggingfacehub 模型 ID 名称
    report_to = "wandb", # 如果您不想使用 wandb，请将此项留空
    run_name = "RAFT_SFT_Take7",
    eval_steps = 5,
    eval_strategy = "steps",
    per_device_train_batch_size = 1,    # 如果量化则小批量
    per_device_eval_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate = 2e-5,
    num_train_epochs = 5,
    # max_steps = 60, # 或设置 num_train_epochs
    save_strategy = "no",
    gradient_checkpointing = True,
    logging_strategy = "steps",
    logging_steps = 5,
    seed = 42,
    optim = "adamw_torch",
    lr_scheduler_type = "cosine",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds, 
    args = training_args,
    dataset_text_field = "text",
    
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/301 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/34 [00:00<?, ? examples/s]

当前内存统计信息

In [29]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A10G. Max memory = 22.184 GB.
1.457 GB of memory reserved.


In [30]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 301 | Num Epochs = 5 | Total steps = 185
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192/1,000,000,000 (1.13% trained)
wandb: Currently logged in as: tituslhy to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
5,1.493000,1.633143
10,1.466600,1.617843
15,1.546300,1.596143
20,1.485900,1.571562
25,1.449800,1.546785
30,1.426500,1.521693
35,1.446800,1.497457
40,1.376700,1.474485
45,1.334400,1.454567
50,1.365500,1.434021


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


使用内存统计

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

637.9309 seconds used for training.
10.63 minutes used for training.
Peak reserved memory = 2.156 GB.
Peak reserved memory for training = 0.699 GB.
Peak reserved memory % of max memory = 9.719 %.
Peak reserved memory for training % of max memory = 3.151 %.


<a name="保存"></a>
### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [ ]:
# 合并到16位
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 合并到4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

[**新**] 要微调并自动导出到 Ollama，请尝试我们的 [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# 保存到8位Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# 记得去 https://huggingface.co/settings/tokens 获取令牌！
# 并将 hf 更改为您的用户名！
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# 保存到多个 GGUF 选项 - 如果您想要多个，速度会更快！
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # 将 hf 更改为您的用户名！
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1. 希望在本地使用 Unsloth？请阅读我们的 [Installation Guide](https://unsloth.ai/docs/get-started/install)，了解有关在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth 的详细信息。
2. 了解如何使用我们的 [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 进行强化学习。
3. 阅读我们的指南和笔记本以了解 [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 和 [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) 模型支持。
4. 浏览我们的 [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) 以查找每个型号的专用指南。
5. 需要推理方面的帮助吗？请阅读我们的 [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment)，了解有关使用 vLLM、llama.cpp、Ollama 等的详细信息。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  <b>此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可</b>
</div>